A Transformer-based recommender with steps: (a) adding positional embeddings to item ID embeddings, (b) a self-attention encoder layer, (c) producing an output embedding for the next-item prediction, and (d) computing a loss against the true next item.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [14]:
class SASRecTransformer(nn.Module):
    """
    Self-Attentive Sequential Recommendation model.
    Predicts the next item in a user's interaction sequence.
    """
    
    def __init__(self, num_items, embedding_dim, num_heads, num_layers, 
                 max_seq_length, dropout_rate=0.1, device='cpu'):
        """
        Args:
            num_items: Total number of items in the catalog
            embedding_dim: Dimension of item embeddings
            num_heads: Number of attention heads
            num_layers: Number of transformer encoder layers
            max_seq_length: Maximum sequence length
            dropout_rate: Dropout probability
            device: Device to place tensors on ('cpu' or 'cuda')
        """
        super(SASRecTransformer, self).__init__()
        
        self.num_items = num_items
        self.embedding_dim = embedding_dim
        self.num_heads = num_heads
        self.num_layers = num_layers
        self.max_seq_length = max_seq_length
        self.dropout_rate = dropout_rate
        self.device = device
        
        # ========== (a) Embedding Layers ==========
        self.item_embedding = nn.Embedding(
            num_items + 1,  # +1 for padding token (0)
            embedding_dim,
            padding_idx=0
        )
        
        self.positional_embedding = nn.Embedding(
            max_seq_length,
            embedding_dim
        )
        
        self.embedding_dropout = nn.Dropout(dropout_rate)
        
        
        # ========== (b) Self-Attention Encoder Layers ==========
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dim_feedforward=embedding_dim * 4,  # FFN hidden dimension. Hidden size to 4× the model/embedding dimension so it gives a richer intermediate representation.
            dropout=dropout_rate,
            activation='relu',
            batch_first=True,  # (batch, seq, features)
            norm_first=True     # Apply LayerNorm before attention/FFN
        )
        
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
            norm=nn.LayerNorm(embedding_dim)
        )
        
        self.dropout = nn.Dropout(dropout_rate)
        
    
    def forward(self, user_sequence, target_item=None):
        """
        Forward pass for training or inference.
        
        Args:
            user_sequence: Tensor of shape [batch_size, seq_length] containing item IDs
            target_item: Tensor of shape [batch_size] containing the true next item (for loss computation)
        
        Returns:
            loss: Cross-entropy loss (if target_item provided)
            predictions: Logits of shape [batch_size, num_items]
            output_embedding: Final user interest embedding [batch_size, embedding_dim]
        """
        batch_size, seq_length = user_sequence.shape
        
        # ========== (a) Add Positional Embeddings ==========
        # Lookup item embeddings
        item_emb = self.item_embedding(user_sequence)
        # item_emb: [batch_size, seq_length, embedding_dim]
        
        # Create position indices
        positions = torch.arange(seq_length, dtype=torch.long, device=self.device)
        pos_emb = self.positional_embedding(positions)
        # pos_emb: [seq_length, embedding_dim]
        
        # Add positional embeddings to item embeddings
        seq_embeddings = item_emb + pos_emb.unsqueeze(0)
        # seq_embeddings: [batch_size, seq_length, embedding_dim]
        
        seq_embeddings = self.embedding_dropout(seq_embeddings)
        
        
        # ========== (b) Self-Attention Encoder Layer(s) ==========
        # Create causal mask (upper triangular matrix)
        # Each position can only attend to itself and previous positions
        causal_mask = torch.triu(
            torch.ones(seq_length, seq_length, device=self.device) * float('-inf'),
            diagonal=1
        )
        
        # Create padding mask (mask out padded positions with 0 item ID)
        padding_mask = (user_sequence == 0)  # True where padding exists
        
        # Apply transformer encoder with causal and padding masks
        encoded = self.transformer_encoder(
            seq_embeddings,
            mask=causal_mask,
            src_key_padding_mask=padding_mask
        )
        # encoded: [batch_size, seq_length, embedding_dim]
        
        
        # ========== (c) Output Embedding for Next-Item Prediction ==========
        # Extract the embedding at the last position (user's final interest)
        # For variable-length sequences, use the actual sequence length before padding
        last_positions = (seq_length - 1) * torch.ones(batch_size, dtype=torch.long, device=self.device)
        output_embedding = encoded[torch.arange(batch_size), last_positions.long()] # This line uses advanced indexing to pick, for each sequence in the batch, the hidden state at a specific timestep
        # output_embedding: [batch_size, embedding_dim]
        
        
        # Compute prediction scores (logits) for all items
        # Dot product between user's final interest and all item embeddings
        logits = torch.matmul(
            output_embedding,
            self.item_embedding.weight[1:].t()  # Exclude padding token
        )
        # logits: [batch_size, num_items]
        
        predictions = F.softmax(logits, dim=-1)
        # predictions: [batch_size, num_items]
        
        
        # ========== (d) Compute Loss Against True Next Item ==========
        loss = None
        if target_item is not None:
            # Cross-entropy loss between predicted logits and true next item
            loss = F.cross_entropy(logits, target_item - 1)  # -1 to account for excluded padding token
        
        return loss, logits, predictions, output_embedding

In [16]:
# ========== Example Usage ==========

if __name__ == "__main__":
    # Hyperparameters
    num_items = 1000
    embedding_dim = 64
    num_heads = 4
    num_layers = 2
    max_seq_length = 50
    batch_size = 32
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # Initialize model
    model = SASRecTransformer(
        num_items=num_items,
        embedding_dim=embedding_dim,
        num_heads=num_heads,
        num_layers=num_layers,
        max_seq_length=max_seq_length,
        dropout_rate=0.1,
        device=device
    )
    model.to(device)
    
    # Create dummy data
    user_sequences = torch.randint(1, num_items + 1, (batch_size, max_seq_length), device=device)
    target_items = torch.randint(1, num_items + 1, (batch_size,), device=device)
    
    # Forward pass
    loss, logits, predictions, output_embedding = model(user_sequences, target_items)
    
    print(f"Loss: {loss.item():.4f}")
    print(f"Logits shape: {logits.shape}")
    print(f"Predictions shape: {predictions.shape}")
    print(f"Output embedding shape: {output_embedding.shape}")

c:\Users\zhong\miniconda3\Lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
c:\Users\zhong\miniconda3\Lib\site-packages\torch\nn\functional.py:6044: UserWarning: Support for mismatched src_key_padding_mask and mask is deprecated. Use same type for both instead.
  warnings.warn(


Loss: 39.6733
Logits shape: torch.Size([32, 1000])
Predictions shape: torch.Size([32, 1000])
Output embedding shape: torch.Size([32, 64])


A small snippet to create positional encoding vectors for positions 0 through 9 for a model with embedding size 16.

In [2]:
# ========== Positional Encoding Example ==========

# Approach 1: Learnable Positional Embeddings (simplest)
num_positions = 10
embedding_size = 16

# Create learnable positional embeddings
pos_embeddings = nn.Embedding(num_positions, embedding_size)
print("Learnable Positional Embeddings shape:", pos_embeddings.weight.shape)
print(pos_embeddings.weight)

# Get positional encodings for positions 0-9
positions = torch.arange(num_positions)
pos_encoding = pos_embeddings(positions)
print("\nPositional Encoding for positions 0-9:")
print(pos_encoding.shape)  # [10, 16]
print(pos_encoding[:3])  # First 3 positions

# ========== Adding to Item Embeddings ==========
# Suppose we have item embeddings for a sequence
batch_size = 2
seq_length = 10
num_items = 100

# Create item embeddings
item_embedding = nn.Embedding(num_items + 1, embedding_size, padding_idx=0)
item_ids = torch.randint(1, num_items + 1, (batch_size, seq_length))
item_emb = item_embedding(item_ids)  # [2, 10, 16]

print("\n--- Adding Positional Encodings to Item Embeddings ---")
print("Item embedding shape:", item_emb.shape)

# Get positional encodings for this sequence length
pos_enc = pos_embeddings(torch.arange(seq_length))  # [10, 16]
print("Positional encoding shape:", pos_enc.shape)

# Add: broadcast positional encoding across batch dimension
combined_embeddings = item_emb + pos_enc.unsqueeze(0)
print("Combined embeddings shape:", combined_embeddings.shape)  # [2, 10, 16]

print("\nFirst item + positional encoding for first example:")
print(combined_embeddings[0, 0])

Learnable Positional Embeddings shape: torch.Size([10, 16])
Parameter containing:
tensor([[ 1.2659e+00, -1.7195e+00,  4.7144e-01,  3.9991e-01, -4.9664e-01,
          4.3086e-01, -1.9072e-01,  1.9980e-01,  6.0241e-02,  1.2592e+00,
         -4.2952e-02, -3.5593e-01, -5.5043e-01,  1.0782e+00,  6.5847e-01,
          8.2112e-01],
        [ 1.9663e+00, -5.6389e-01,  1.4317e+00, -5.5224e-01,  2.8124e+00,
          9.2977e-01, -1.2359e-01, -6.9860e-01, -1.4914e-01, -8.1771e-01,
          9.7219e-02, -7.6878e-01,  2.9738e-02,  7.5217e-01, -3.6053e-01,
         -1.2309e+00],
        [-7.7534e-01, -1.3185e+00, -9.0055e-01, -1.5660e+00,  1.3960e-02,
         -1.4705e-01,  1.0518e+00, -2.0254e-01, -1.7532e+00,  4.8503e-01,
          8.6435e-01, -1.6374e+00,  1.3043e+00, -1.3569e+00,  6.8355e-01,
          1.5105e-01],
        [-2.1140e+00, -6.9904e-01,  4.6324e-01,  3.5931e-01, -2.9830e-01,
         -2.4395e-01, -4.8364e-02,  8.5044e-01, -3.8323e-01, -4.8379e-02,
         -1.3196e+00,  6.8511e-02, 

In implementing a Transformer for recommendation, we need to prevent the model from cheating by peeking ahead. In SASRec (unidirectional setting), this is done with a causal mask in self-attention. Implement this causal masking in the attention mechanism (a brief matrix example of a mask for sequence length 5).

In [24]:
# ========== Causal Masking in Self-Attention ==========

# Why causal masking?
# In SASRec, we want to prevent the model from "cheating" by looking at future items.
# At prediction time, the model only has access to past interactions.
# So during training, we mask out all future positions in the attention mechanism.

print("=" * 60)
print("CAUSAL MASK FOR SEQUENCE LENGTH 5")
print("=" * 60)

seq_length = 5

# Create causal mask: each position can attend to itself and previous positions only
# Upper triangular matrix with -inf for forbidden positions (future positions)
causal_mask = torch.triu(
    torch.ones(seq_length, seq_length) * float('-inf'),
    diagonal=1  # diagonal=1 means elements ABOVE the main diagonal get -inf
)

print("\nCausal Mask Matrix (5x5):")
print("(Rows = Query positions, Columns = Key positions)")
print("(inf means 'cannot attend'; 0 means 'can attend')\n")
print(causal_mask)

print("\n--- Interpretation ---")
print("Position 0 can attend to: [0]")
print("Position 1 can attend to: [0, 1]")
print("Position 2 can attend to: [0, 1, 2]")
print("Position 3 can attend to: [0, 1, 2, 3]")
print("Position 4 can attend to: [0, 1, 2, 3, 4]")

print("\n--- How it works in attention ---")
print("1. Compute attention scores: Q @ K^T (shape: [5, 5])")
print("2. Add causal mask: scores + causal_mask")
print("3. Apply softmax: The -inf values become 0 after softmax")
print("   (softmax(-inf) = 0, so those positions have zero weight)")

# Example with dummy attention scores
print("\n" + "=" * 60)
print("EXAMPLE: Dummy Attention Scores Before Masking")
print("=" * 60)

# Simulate attention scores (e.g., from Q @ K^T)
attention_scores = torch.randn(seq_length, seq_length)
print("\nRaw attention scores (5x5):")
print(attention_scores.round(decimals=2))

# Apply causal mask
masked_scores = attention_scores + causal_mask
print("\nAfter adding causal mask (notice -inf in upper triangle):")
print(masked_scores.round(decimals=2))

# Apply softmax
attention_weights = F.softmax(masked_scores, dim=-1)
print("\nAfter softmax (notice 0s where masked):")
print(attention_weights.round(decimals=3))

print("\n--- Verification ---")
print("Notice position 0's attention weights:", attention_weights[0].round(decimals=3))
print("(All weight on position 0 only)")
print("\nNotice position 4's attention weights:", attention_weights[4].round(decimals=3))
print("(Weights distributed across all positions 0-4)")


CAUSAL MASK FOR SEQUENCE LENGTH 5

Causal Mask Matrix (5x5):
(Rows = Query positions, Columns = Key positions)
(inf means 'cannot attend'; 0 means 'can attend')

tensor([[0., -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0.]])

--- Interpretation ---
Position 0 can attend to: [0]
Position 1 can attend to: [0, 1]
Position 2 can attend to: [0, 1, 2]
Position 3 can attend to: [0, 1, 2, 3]
Position 4 can attend to: [0, 1, 2, 3, 4]

--- How it works in attention ---
1. Compute attention scores: Q @ K^T (shape: [5, 5])
2. Add causal mask: scores + causal_mask
3. Apply softmax: The -inf values become 0 after softmax
   (softmax(-inf) = 0, so those positions have zero weight)

EXAMPLE: Dummy Attention Scores Before Masking

Raw attention scores (5x5):
tensor([[ 0.7300, -0.8500,  1.3000,  1.7300,  0.5100],
        [-0.4800,  0.2700, -0.3100, -0.3000,  0.6100],
        [ 0.9000,  1.5500,  0.

Incorporate additional features (like item category or timestamp of interaction) into a Transformer-based sequential model. Modify the input or the architecture to include this information (for example, by modifying the input embedding to be a sum of item embedding + category embedding + time embedding, or by concatenating features and projecting, etc).

**Incorporating extra features (category, timestamp, etc.) into a Transformer**

- **Sum embeddings (additive):** For each position, embed item ID (`E_item`), category (`E_cat`), and discretized time bucket (`E_time`), then sum: `x = E_item + E_cat + E_time + E_pos`. Simple and keeps shape `[B, L, d]`.
- **Concatenate + project:** Concatenate all embeddings and continuous features: `concat = [E_item || E_cat || E_time || f_continuous]` (shape `[B, L, d_total]`), then project: `x = W_proj * concat + b` to `[B, L, d_model]`, finally add positional `E_pos`. More flexible for mixed feature sizes.
- **Continuous time encoding:** For raw timestamps or deltas, either bucketize (learned embedding) or apply a small MLP on normalized time gaps, then sum/concat into the token representation.
- **Feature-specific dropout/gating:** Optionally apply dropout or a learned gate per feature block before combining to avoid over-reliance on any single feature.
- **Side loss for auxiliary features:** If you predict side labels (e.g., next category), add an auxiliary head on the same Transformer outputs; this regularizes shared representations.
- **Masking stays the same:** Causal mask still applies to the combined token representations; only the input projection/embedding block changes.

**Minimal code sketch (concat + project):**
```python
class FeatureFusion(nn.Module):
    def __init__(self, d_item, d_cat, d_time, d_cont, d_model):
        super().__init__()
        self.item_emb = nn.Embedding(num_items + 1, d_item, padding_idx=0)
        self.cat_emb = nn.Embedding(num_cats + 1, d_cat, padding_idx=0)
        self.time_emb = nn.Embedding(num_time_buckets + 1, d_time, padding_idx=0)
        self.proj = nn.Linear(d_item + d_cat + d_time + d_cont, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

    def forward(self, item_ids, cat_ids, time_ids, cont_feats):
        # cont_feats: [B, L, d_cont] continuous features (e.g., normalized time gaps)
        x = torch.cat([
            self.item_emb(item_ids),
            self.cat_emb(cat_ids),
            self.time_emb(time_ids),
            cont_feats,
        ], dim=-1)
        x = self.proj(x) + self.pos_emb(torch.arange(x.size(1), device=x.device)).unsqueeze(0)
        return x  # feed into Transformer with causal masking
```
